In [ ]:
import os
import sys

# Clone the repository
if not os.path.exists('/content/CTAB-GAN'):
    !git clone https://github.com/Team-TUD/CTAB-GAN.git

# Change directory and add to path
os.chdir('/content/CTAB-GAN')
sys.path.insert(0, '/content/CTAB-GAN')

# Install dependencies with compatible versions
!pip install dython==0.7.6 --quiet

# Now import
from model.ctabgan import CTABGAN

In [ ]:
!git clone https://github.com/JoeNissen/ml-final-project

In [ ]:
%cd ml-final-project

In [ ]:
# Install SDV library
!pip install sdv --quiet

In [ ]:
# Uninstall and reinstall scikit-learn completely
!pip uninstall scikit-learn -y
!pip install scikit-learn==1.5.1

In [ ]:
import sys
sys.path.insert(0, "../")

!pip install rdt

#from ctgan.synthesizers.ctgan import CTGANSynthesizer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

from src.data_loader import load_adult_data
from src.utils import *

# Get Data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

# Load adult.csv assuming no header
D_adult = pd.read_csv("adult.csv", header=None)

# Define standard column names for the adult dataset
column_names = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
                'marital-status', 'occupation', 'relationship', 'race', 'sex',
                'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

# Assign these column names to the DataFrame
D_adult.columns = column_names

# Drop the first row, which contains the original string headers now incorrectly part of the data
D_adult = D_adult.iloc[1:].reset_index(drop=True)

# Convert numeric columns to appropriate types after dropping the header row
numeric_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
for col in numeric_cols:
    D_adult[col] = pd.to_numeric(D_adult[col], errors='coerce')

# Rename 'income' to 'y'
D_adult = D_adult.rename(columns={'income': 'y'})

# Extract target variable 'y' and features 'X'
y = D_adult["y"]
X = D_adult.drop(columns=["y"])

seed = 0
X_train, X_test = train_test_split(D_adult, test_size=0.6, random_state=seed)

# Train base models

In [ ]:
from copy import deepcopy
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

model_dict = {
    "mlp": MLPClassifier(),
    "knn": KNeighborsClassifier(),
    "dt": DecisionTreeClassifier(),
    "rf": RandomForestClassifier(),
    "gbc": GradientBoostingClassifier(),
}

trained_model_dict = {}

# Make a copy to avoid modifying the original X_train which might be needed later
X_train_processed = X_train.copy()

# Identify categorical columns (excluding the target 'y')
categorical_cols_features = [
    'workclass', 'education', 'marital-status', 'occupation',
    'relationship', 'race', 'sex', 'native-country'
]

# Apply one-hot encoding to categorical features
X_train_processed = pd.get_dummies(X_train_processed, columns=categorical_cols_features, drop_first=True)

# Encode the target variable 'y'
le = LabelEncoder()
X_train_processed['y'] = le.fit_transform(X_train_processed['y'])

for model in model_dict.keys():
    clf = model_dict[model]
    # Use the processed data for fitting
    clf.fit(X_train_processed.drop("y", axis=1), X_train_processed["y"])

    trained_model_dict[model] = deepcopy(clf)


# Train Generative model

In [ ]:
import os
import sys

# Navigate to a different directory
os.chdir('/content')

# Remove all CTAB-GAN related paths
sys.path = [p for p in sys.path if 'CTAB-GAN' not in p]

# Rename the CTAB-GAN folder to avoid conflicts
if os.path.exists('/content/CTAB-GAN'):
    os.rename('/content/CTAB-GAN', '/content/CTAB-GAN-BACKUP')

# Clear any cached modules
modules_to_remove = [k for k in list(sys.modules.keys()) if 'ctgan' in k.lower() or 'ctab' in k.lower()]
for module in modules_to_remove:
    del sys.modules[module]
    print(f"Cleared: {module}")


# Now import SDV
from sdv.single_table import TVAESynthesizer

In [ ]:
from sdv.single_table import TVAESynthesizer

# Define discrete/categorical columns
discrete_columns = [
    "workclass", "education", "education-num", "marital-status",
    "occupation", "relationship", "race", "sex", "native-country", "y"
]

# Initialize TVAE with similar configuration to CTGAN
syn_model = TVAESynthesizer(
    epochs=300,
    cuda=True,
    batch_size=500,
    compress_dims=(128, 128),
    decompress_dims=(128, 128),
)

# Set random seed
seed_everything(seed)

# Train the model
print("Training TVAE...")
syn_model.fit(X_train)

# Generate synthetic samples
print("Generating synthetic data...")
synthetic_data = syn_model.sample(num_rows=len(X_train))

print(f"Original data shape: {X_train.shape}")
print(f"Synthetic data shape: {synthetic_data.shape}")

In [ ]:
from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata

# Create metadata from data
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(X_train)

# Initialize TVAE with metadata
syn_model = TVAESynthesizer(
    metadata=metadata,
    epochs=300,
    cuda=True,
    batch_size=500,
    compress_dims=(128, 128),
    decompress_dims=(128, 128),
)

# Set random seed
seed_everything(seed)

# Train the model
print("Training TVAE...")
syn_model.fit(X_train)

# Generate synthetic samples
print("Generating synthetic data...")
synthetic_data = syn_model.sample(num_rows=len(X_train))

print(f"Original data shape: {X_train.shape}")
print(f"Synthetic data shape: {synthetic_data.shape}")

In [ ]:
from copy import deepcopy
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Prepare synthetic data the same way as original training process
synthetic_data_processed = synthetic_data.copy()

# Apply one-hot encoding to categorical features
categorical_cols_features = [
    'workclass', 'education', 'marital-status', 'occupation',
    'relationship', 'race', 'sex', 'native-country'
]

synthetic_data_processed = pd.get_dummies(
    synthetic_data_processed,
    columns=categorical_cols_features,
    drop_first=True
)

# Encode the target variable 'y'
le = LabelEncoder()
synthetic_data_processed['y'] = le.fit_transform(synthetic_data_processed['y'])

print(f"Processed synthetic data shape: {synthetic_data_processed.shape}")

# Train models on synthetic data
synthetic_trained_models = {}

for model_name in model_dict.keys():
    clf = deepcopy(model_dict[model_name])
    clf.fit(
        synthetic_data_processed.drop("y", axis=1),
        synthetic_data_processed["y"]
    )
    synthetic_trained_models[model_name] = clf


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Prepare X_test the same way as X_train was processed
X_test_processed = X_test.copy()

# Apply one-hot encoding to categorical features
categorical_cols_features = [
    'workclass', 'education', 'marital-status', 'occupation',
    'relationship', 'race', 'sex', 'native-country'
]

X_test_processed = pd.get_dummies(
    X_test_processed,
    columns=categorical_cols_features,
    drop_first=True
)

# Encode the target variable 'y'
le = LabelEncoder()
le.fit(X_train['y'])  # Fit on train to ensure consistent encoding
X_test_processed['y'] = le.transform(X_test_processed['y'])

print(f"Test set processed shape: {X_test_processed.shape}")

# Now evaluate both real-trained and synthetic-trained models
print("\n" + "="*60)
print("PERFORMANCE COMPARISON: Real vs TVAE Synthetic Data")
print("="*60)

for model_name in model_dict.keys():
    # Real-trained model (trained on real X_train)
    real_model = trained_model_dict[model_name]

    # Align test data with real training data columns
    train_cols_real = X_train_processed.drop("y", axis=1).columns
    X_test_for_real = X_test_processed.drop("y", axis=1).copy()

    # Add missing columns
    for col in train_cols_real:
        if col not in X_test_for_real.columns:
            X_test_for_real[col] = 0

    # Keep only training columns
    X_test_for_real = X_test_for_real[train_cols_real]

    real_pred = real_model.predict(X_test_for_real)
    real_acc = accuracy_score(X_test_processed["y"], real_pred)

    # TVAE synthetic-trained model
    synthetic_model = synthetic_trained_models[model_name]

    # Align test data with synthetic training data columns
    train_cols_synthetic = synthetic_data_processed.drop("y", axis=1).columns
    X_test_for_synthetic = X_test_processed.drop("y", axis=1).copy()

    # Add missing columns
    for col in train_cols_synthetic:
        if col not in X_test_for_synthetic.columns:
            X_test_for_synthetic[col] = 0

    # Keep only training columns
    X_test_for_synthetic = X_test_for_synthetic[train_cols_synthetic]

    synthetic_pred = synthetic_model.predict(X_test_for_synthetic)
    synthetic_acc = accuracy_score(X_test_processed["y"], synthetic_pred)

    print(f"\n{model_name.upper()}:")
    print(f"  Real data accuracy:      {real_acc:.4f}")
    print(f"  TVAE synthetic accuracy: {synthetic_acc:.4f}")
    print(f"  Difference:              {abs(real_acc - synthetic_acc):.4f}")

print("\n" + "="*60)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde, norm

# Define rejection_sample function from the notebook
def rejection_sample(D: pd.DataFrame, mean: float, std: float, feat_id: list):
    """
    Performs rejection sampling to shift the distribution of a specified numeric feature.
    """
    if not feat_id or not isinstance(feat_id, list) or len(feat_id) == 0:
        raise ValueError("feat_id must be a list containing the index of the feature column.")

    col_idx = feat_id[0]
    original_column_data = D.iloc[:, col_idx].values

    # Create KDE for the original distribution (p)
    p_orig = gaussian_kde(original_column_data.reshape(1, -1))

    # Define the target distribution (p_shifted) - a normal distribution
    p_shifted_func = lambda x: norm.pdf(x, loc=mean, scale=std)

    # Estimate M (maximum ratio)
    data_min, data_max = original_column_data.min(), original_column_data.max()
    sample_points = np.linspace(data_min - 3*std, data_max + 3*std, 1000)

    p_orig_at_points = p_orig(sample_points.reshape(1, -1))[0]
    p_shifted_at_points = p_shifted_func(sample_points)

    epsilon = 1e-9
    ratio_estimates = np.where(p_orig_at_points > epsilon, p_shifted_at_points / p_orig_at_points, 0)

    M = np.max(ratio_estimates) * 1.1
    if M == 0:
        M = 1.0

    accepted_samples_data = []
    target_num_samples = 10000

    if len(D) == 0:
        return np.array([])

    num_candidates_to_generate = target_num_samples * max(int(M * 2), 10)
    candidate_indices = np.random.choice(len(D), size=num_candidates_to_generate, replace=True)

    for i in range(num_candidates_to_generate):
        if len(accepted_samples_data) >= target_num_samples:
            break

        idx = candidate_indices[i]
        candidate_row = D.iloc[idx]
        x_val = candidate_row.iloc[col_idx]

        p_s = p_shifted_func(x_val)
        p_o = p_orig(np.array([[x_val]]))[0]

        if p_o > epsilon:
            acceptance_prob = p_s / (M * p_o)
        else:
            acceptance_prob = 0

        if np.random.rand() < acceptance_prob:
            accepted_samples_data.append(candidate_row.values)

    return np.array(accepted_samples_data)


In [ ]:
ys_mlp_tvae_all = []
ys_knn_tvae_all = []
ys_dt_tvae_all = []
ys_rf_tvae_all = []
ys_gbc_tvae_all = []

# Use the same parameters from the CTGAN notebook
metric = "age"
data = X_train[metric]
mean, std = np.mean(data), np.std(data)
n_range = 10
n_std = 1 * std

for i in range(2):  # Same as CTGAN notebook (2 iterations)

    ys_mlp_tmp = []
    ys_knn_tmp = []
    ys_dt_tmp = []
    ys_rf_tmp = []
    ys_gbc_tmp = []

    # Generate synthetic data from TVAE (instead of CTGAN)
    shift_df = syn_model.sample(num_rows=10000)

    xs = list(
        np.arange(
            mean - n_std, mean + n_std, ((mean + n_std) - (mean - n_std)) / n_range
        )
    )

    for shift_mean in np.arange(
        mean - n_std, mean + n_std, ((mean + n_std) - (mean - n_std)) / n_range
    ):
        reject_df = rejection_sample(
            D=shift_df, mean=shift_mean, std=std / 2, feat_id=[0]
        )
        if len(reject_df) == 0:
            continue

        test_df = pd.DataFrame(reject_df, columns=X_test.columns)
        real_tester = test_df

        real_tester_processed = real_tester.copy()
        categorical_cols_features_test = [col for col in X_test.columns if X_test[col].dtype == 'object' and col != 'y']
        real_tester_processed = pd.get_dummies(real_tester_processed, columns=categorical_cols_features_test, drop_first=True)

        if 'y' in real_tester_processed.columns:
            real_tester_processed['y'] = le.transform(real_tester_processed['y'])

        # Align columns
        train_cols = X_train_processed.drop("y", axis=1).columns
        missing_cols = set(train_cols) - set(real_tester_processed.drop("y", axis=1).columns)
        for c in missing_cols:
            real_tester_processed[c] = 0

        extra_cols = set(real_tester_processed.drop("y", axis=1).columns) - set(train_cols)
        real_tester_processed = real_tester_processed.drop(columns=list(extra_cols))
        real_tester_processed = real_tester_processed[train_cols.tolist() + ['y'] if 'y' in real_tester_processed.columns else train_cols.tolist()]

        X_for_prediction = real_tester_processed.drop("y", axis=1)
        y_true = real_tester_processed["y"]

        for model in model_dict.keys():
            clf = trained_model_dict[model]
            y_pred = clf.predict(X_for_prediction)

            if model == "mlp":
                ys_mlp_tmp.append(accuracy_score(y_true, y_pred))
            if model == "knn":
                ys_knn_tmp.append(accuracy_score(y_true, y_pred))
            if model == "dt":
                ys_dt_tmp.append(accuracy_score(y_true, y_pred))
            if model == "rf":
                ys_rf_tmp.append(accuracy_score(y_true, y_pred))
            if model == "gbc":
                ys_gbc_tmp.append(accuracy_score(y_true, y_pred))

    ys_mlp_tvae_all.append(ys_mlp_tmp)
    ys_knn_tvae_all.append(ys_knn_tmp)
    ys_dt_tvae_all.append(ys_dt_tmp)
    ys_rf_tvae_all.append(ys_rf_tmp)
    ys_gbc_tvae_all.append(ys_gbc_tmp)


In [ ]:
# Run the oracle/test rejection sampling (from the CTGAN notebook)
yr_mlp = []
yr_knn = []
yr_dt = []
yr_rf = []
yr_gbc = []

xr = list(
    np.arange(mean - n_std, mean + n_std, ((mean + n_std) - (mean - n_std)) / n_range)
)

for shift_mean in np.arange(
    mean - n_std, mean + n_std, ((mean + n_std) - (mean - n_std)) / n_range
):
    reject_df = rejection_sample(D=X_test, mean=shift_mean, std=std / 2, feat_id=[0])
    if len(reject_df) == 0:
        continue
    test_df = pd.DataFrame(reject_df, columns=X_test.columns)
    real_tester = test_df

    # Preprocessing
    real_tester_processed = real_tester.copy()
    categorical_cols_features_test = [col for col in X_test.columns if X_test[col].dtype == 'object' and col != 'y']
    real_tester_processed = pd.get_dummies(real_tester_processed, columns=categorical_cols_features_test, drop_first=True)

    if 'y' in real_tester_processed.columns:
        real_tester_processed['y'] = le.transform(real_tester_processed['y'])

    # Align columns
    train_cols = X_train_processed.drop("y", axis=1).columns
    missing_cols = set(train_cols) - set(real_tester_processed.drop("y", axis=1).columns)
    for c in missing_cols:
        real_tester_processed[c] = 0

    extra_cols = set(real_tester_processed.drop("y", axis=1).columns) - set(train_cols)
    real_tester_processed = real_tester_processed.drop(columns=list(extra_cols))
    real_tester_processed = real_tester_processed[train_cols.tolist() + ['y'] if 'y' in real_tester_processed.columns else train_cols.tolist()]

    X_for_prediction = real_tester_processed.drop("y", axis=1)
    y_true = real_tester_processed["y"]

    for model in model_dict.keys():
        clf = trained_model_dict[model]
        y_pred = clf.predict(X_for_prediction)

        if model == "mlp":
            yr_mlp.append(accuracy_score(y_true, y_pred))
        if model == "knn":
            yr_knn.append(accuracy_score(y_true, y_pred))
        if model == "dt":
            yr_dt.append(accuracy_score(y_true, y_pred))
        if model == "rf":
            yr_rf.append(accuracy_score(y_true, y_pred))
        if model == "gbc":
            yr_gbc.append(accuracy_score(y_true, y_pred))


In [ ]:
# Compute TVAE results using the same format as CTGAN
ids = np.where((X_train[metric] > xs[0]) & (X_train[metric] < xs[-1]))
quantiles = X_train[metric].iloc[ids].quantile([0.25, 0.5, 0.75]).values
q1 = np.array(xs) < quantiles[0]
q2 = (np.array(xs) > quantiles[0]) & (np.array(xs) < quantiles[2])
q3 = np.array(xs) > quantiles[2]

results_tvae = {}

q1_dict = {}
q1_dict["Error 3S (TVAE)"] = np.mean(np.abs(np.mean(ys_rf_tvae_all, axis=0) - yr_rf)[q1])
q1_dict["Error 3S (CTGAN)"] = np.float64(0.10230000000000002)

q2_dict = {}
q2_dict["Error 3S (TVAE)"] = np.mean(np.abs(np.mean(ys_rf_tvae_all, axis=0) - yr_rf)[q2])
q2_dict["Error 3S (CTGAN)"] = np.float64(0.1009125)

q3_dict = {}
q3_dict["Error 3S (TVAE)"] = np.mean(np.abs(np.mean(ys_rf_tvae_all, axis=0) - yr_rf)[q3])
q3_dict["Error 3S (CTGAN)"] = np.float64(0.10873333333333333)

results_tvae["Q1"] = q1_dict
results_tvae["Q2"] = q2_dict
results_tvae["Q3"] = q3_dict

avg_dict = {}
threeS_err_tvae = np.abs(np.mean(ys_rf_tvae_all, axis=0) - yr_rf)
avg_dict["Error 3S (TVAE)"] = np.mean(threeS_err_tvae)
avg_dict["Error 3S (CTGAN)"] = np.float64(0.103675)

results_tvae["avg"] = avg_dict

print("\n" + "="*60)
print("DISTRIBUTION SHIFT ROBUSTNESS: TVAE vs CTGAN")
print("="*60)

for key, val in results_tvae.items():
    print(f"\n{key}:")
    for metric, error in val.items():
        print(f"  {metric}: {error:.5f}")
    print()  # Extra newline between sections